# Alion quickstart: company data in Python

[Alion](https://alion.io/developers) is a live graph of 3.7 million companies: what each one does, where it operates, its stack, funding rounds and investors, owners, dated events, public contracts, and verified job postings with pay statistics.

The first part runs **without an account**. The second needs an API key (a key comes with $1 of free data every month).

In [ ]:
%pip install -q "alion[pandas]"

## 1. Free: a prospect list

In [ ]:
from alion import Alion, to_pandas

alion = Alion(api_key="")  # no key: the open address: free, 300 calls a day
rows = alion.search_companies(industry="Fintech", country="DE", startup=True, hiring=True, limit=20)
to_pandas(rows)[["name", "domain", "size_band", "founded_year", "open_jobs", "url"]]

## 2. Free: lookalikes, matching, hiring

In [ ]:
to_pandas(alion.similar_companies(company="stripe.com", limit=10))[["name", "domain", "hq_country", "size_band"]]

In [ ]:
alion.resolve_company(query="Trade Republic")["match"]

In [ ]:
jobs = to_pandas(alion.search_jobs(company="stripe.com", limit=20))
jobs["role_family"].value_counts()

## 3. With a key: enrichment, money, events

Create a key at [alion.io/developers/keys](https://alion.io/developers/keys). In Colab, put it in the Secrets panel as `ALION_API_KEY`.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["ALION_API_KEY"] = userdata.get("ALION_API_KEY")
except Exception:
    pass

paid = Alion(max_credits=20)  # refuses anything that could pass 20 credits ($2)

In [ ]:
# $0.01 a company + $0.005 a slice; unknown domains are free
res = paid.enrich_companies(["stripe.com", "adyen.com", "mollie.com"], slices="funding,hiring")
to_pandas(res)

In [ ]:
# the last week's funding rounds in Germany, $0.005 an event
events = list(paid.events(type="funding_round", country="DE", since="7d", follow=False))
to_pandas(events)[["date", "company.name", "company.domain", "type"]]

In [ ]:
print(f"spent {paid.credits_spent} credits")

More: [examples](https://github.com/alion-io/alion-sdk/tree/main/examples), [docs](https://alion.io/developers/docs), [pricing](https://alion.io/developers/pricing).